# Day 51 — Advanced feature engineering: target encoding & leakage prevention
Objectives:
- Understand target/mean encoding for high-cardinality categoricals.
- Prevent leakage with KFold schemes.
- Integrate into sklearn Pipelines.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','embarked','fare','age'])
X = df[['sex','class','embarked','fare','age']]
y = df['survived']
Xtr,Xte,ytr,yte = train_test_split(X,y, stratify=y, random_state=42)


## KFold target encoding utility (no leakage)
For each fold, compute means on train folds and apply to val fold only.

In [ ]:
def kfold_target_encode(cat: pd.Series, y: pd.Series, n_splits=5, smoothing=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    out = pd.Series(index=cat.index, dtype=float)
    global_mean = y.mean()
    for tidx, vidx in skf.split(cat, y):
        trc, trY = cat.iloc[tidx], y.iloc[tidx]
        means = trY.groupby(trc).mean()
        counts = trY.groupby(trc).size()
        smooth = (means * counts + global_mean * smoothing) / (counts + smoothing)
        out.iloc[vidx] = cat.iloc[vidx].map(smooth).fillna(global_mean)
    return out.fillna(global_mean)

Xe = Xtr.copy()
for col in ['sex','class','embarked']:
    Xe[col + '_te'] = kfold_target_encode(Xtr[col], ytr)
Xe[['sex_te','class_te','embarked_te']].head()


## Compare baseline One-Hot vs Target Encoding features
(Demonstration: combine OHE for small cats + TE for high-cardinality if present.)

In [ ]:
from sklearn.metrics import roc_auc_score
# Baseline OHE
ohe = ColumnTransformer([('ohe', OneHotEncoder(handle_unknown='ignore'), ['sex','class','embarked'])], remainder='passthrough')
pipe_ohe = Pipeline([('pre', ohe), ('clf', LogisticRegression(max_iter=1000))])
pipe_ohe.fit(Xtr,ytr); auc_ohe = roc_auc_score(yte, pipe_ohe.predict_proba(Xte)[:,1])
# TE approach
Xtr_te = Xtr.copy(); Xte_te = Xte.copy()
for c in ['sex','class','embarked']:
    Xtr_te[c+'_te'] = kfold_target_encode(Xtr[c], ytr)
    # map train means to test
    m = pd.concat([Xtr[c], ytr], axis=1).groupby(c)['survived'].mean()
    Xte_te[c+'_te'] = Xte[c].map(m).fillna(ytr.mean())
cols = ['fare','age','sex_te','class_te','embarked_te']
clf = LogisticRegression(max_iter=1000).fit(Xtr_te[cols], ytr)
auc_te = roc_auc_score(yte, clf.predict_proba(Xte_te[cols])[:,1])
auc_ohe, auc_te


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — out-of-fold target encoding, smoothing, and transform-time unknowns

### Mental model

Target encoding replaces a category with a statistic of the target.
Computing that statistic from the row's own target leaks the answer.
Training rows therefore need **out-of-fold** encodings: each row is
transformed by a map learned from other folds only.

Validation/test rows use a map learned from the corresponding training
data. Rare categories are noisy, so smoothing shrinks their mean toward
the training prior. Missing and unseen categories need an explicit
fallback. Temporal or grouped data requires a compatible split rather
than ordinary shuffled folds.

### Read the API before running it

- **`groupby(category)[target].agg(['mean', 'count'])`:** builds category evidence only from the allowed training subset.
- **`(count * mean + strength * prior) / (count + strength)`:** smooths low-support categories toward the training-wide prior.
- **out-of-fold transform:** fits one map per training fold and writes values only to that fold's held-out rows.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — show why a one-row category leaks perfectly

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Category identity is available at prediction time, but the current row's target is not.

In [ ]:
import pandas as pd

category = pd.Series(["common", "common", "only_zero", "only_one"])
target = pd.Series([0, 1, 0, 1], dtype=float)
full_map = target.groupby(category).mean()
leaked = category.map(full_map)
print(pd.DataFrame({"category": category, "target": target, "leaked": leaked}))
assert leaked.iloc[2] == target.iloc[2]
assert leaked.iloc[3] == target.iloc[3]

**Expected observation:** Singleton categories receive their own targets exactly, creating a feature that memorizes training labels.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — smooth known categories and fall back for unknowns

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Using one shared prior for missing/unseen categories matches the model contract.

In [ ]:
import pandas as pd

train_category = pd.Series(["a", "a", "a", "b"])
train_target = pd.Series([1, 1, 0, 1], dtype=float)
prior = train_target.mean()
stats = train_target.groupby(train_category).agg(["mean", "count"])
strength = 3.0
smoothed = (
    stats["mean"] * stats["count"] + prior * strength
) / (stats["count"] + strength)
new_category = pd.Series(["a", "b", "unseen", None])
encoded = new_category.map(smoothed).fillna(prior)
print({"prior": prior, "map": smoothed.to_dict(),
       "encoded": encoded.tolist()})
assert encoded.iloc[2] == prior and encoded.iloc[3] == prior

**Expected observation:** The low-support `b` estimate is pulled toward the prior; unseen and missing values use the declared prior.

### Debugging and practice ramp

**Common mistake:** Building one full-training target map and applying it back to those same training rows.

**Diagnostic:** Attach source fold and map-training row IDs to a tiny example; assert every encoded training row is absent from its map's target aggregation.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define out-of-fold target encoding, smoothing, and transform-time unknowns in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not use target encoding until the split unit, unseen fallback, smoothing strength, and fit/transform boundaries are testable.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Add K-fold target encoding to a scikit-learn pipeline through a custom
   transformer or `FunctionTransformer`.

**Verify:** For task `Add K-fold target encoding to a scikit-learn pipeline through a custom`, show the relevant row/group/time identities and assert the training and evaluation information boundaries are disjoint.






2. Add an appropriate prior and smoothing; experiment with `n_splits`.

**Verify:** For task `Add an appropriate prior and smoothing; experiment with nsplits`, show the relevant row/group/time identities and assert the training and evaluation information boundaries are disjoint.






3. Compare ROC AUC with one-hot encoding across multiple seeded train/test
   splits.

**Verify:** For task `Compare ROC AUC with one-hot encoding across multiple seeded train/test`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







### Progressive hints

1. A robust custom transformer needs distinct fitting and transform behavior.
   During training, generate out-of-fold values; for new rows, use a mapping fit
   only on training data. Write down index-alignment rules first.
2. Blend a category mean with the global prior using its support count. Test an
   unseen category and a one-row category deliberately.
3. Reuse each split for both methods and report score differences per seed, not
   only the best run.

The notebook's `kfold_target_encode` is an instructional utility, not a
drop-in production transformer. Its Series indexes and positional fold indexes
must remain aligned.

### Additional mastery practice

Implement target-derived features with row-level lineage. Training encodings must be out-of-fold; validation, test, and future rows use mappings fitted only on prior data.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Out-of-fold invariant:** Create a unique category for every training row and show that a leaky full-data target mean reproduces each label. Then prove that your out-of-fold encoder falls back to the prior instead.
   **Progressive hint:** For a category absent from the fold's training partition, there is no valid category statistic; use the fold training prior.

**Verify:** For task `Out-of-fold invariant: Create a unique category for every training row and show that a leaky...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







5. **Unknown and missing categories:** Define distinct policies for a missing category, an unseen category, and a known category with one observation. Write tests for all three.
   **Progressive hint:** Normalize missing values to an explicit sentinel if missingness is a category; unseen categories generally receive the training global prior.

**Verify:** For task `Unknown and missing categories: Define distinct policies for a missing category, an unseen ca...`, produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







6. **Temporal leakage:** Design target encoding for timestamped events where later labels cannot inform earlier rows. Compare random K-fold encoding with an expanding-time implementation.
   **Progressive hint:** Sort by event time and compute each row's category statistics from strictly earlier labeled rows; handle ties deliberately.

**Verify:** For task `Temporal leakage: Design target encoding for timestamped events where later labels cannot inf...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Out-of-fold invariant


# Practice 5 — Unknown and missing categories


# Practice 6 — Temporal leakage
